# Big Mart Sales Prediction — Beginner ML Notebook

In this notebook I build a simple machine learning solution for the Big Mart Sales Prediction dataset.

The goal is to predict **Item_Outlet_Sales** using product and store information.

I keep the work simple because this is a beginner-level regression exercise.

## 1. Research Question

**Question:** Can I predict the sales of a product in a store using the available product and outlet features?

This is a **regression problem**, because the target value is a number: `Item_Outlet_Sales`.

In [1]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42

## 2. Load the Data

The file `train.csv` must be in the same folder as this notebook.

In [2]:
# Load dataset
train = pd.read_csv("train.csv")

print("Dataset shape:", train.shape)
train.head()

Dataset shape: (8523, 12)


,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


## 3. First Look at the Dataset

Here I check the column names, missing values, and the target variable.

In [3]:
print("Columns:")
print(train.columns.tolist())

print("
Missing values:")
print(train.isna().sum().sort_values(ascending=False))

print("
Target summary:")
print(train["Item_Outlet_Sales"].describe())

SyntaxError: unterminated string literal (detected at line 4) (2029272261.py, line 4)

## 4. Simple Visual Exploration

I plot the target variable and one important numeric feature.

In [ ]:
plt.figure(figsize=(7, 4))
train["Item_Outlet_Sales"].hist(bins=30)
plt.title("Distribution of Item Outlet Sales")
plt.xlabel("Sales")
plt.ylabel("Count")
plt.show()

plt.figure(figsize=(7, 4))
plt.scatter(train["Item_MRP"], train["Item_Outlet_Sales"], alpha=0.4)
plt.title("Item MRP vs Sales")
plt.xlabel("Item MRP")
plt.ylabel("Item Outlet Sales")
plt.show()

## 5. Simple Preprocessing

To keep the notebook beginner-friendly, I use a simple approach:

- Drop ID columns because they are identifiers, not direct numeric features.
- Fill missing numeric values with the median.
- Fill missing categorical values with the most common value.
- Convert categorical columns into numbers using one-hot encoding.

In [ ]:
# Separate target and features
y = train["Item_Outlet_Sales"]
X = train.drop(columns=["Item_Outlet_Sales"])

# Drop identifier columns
X = X.drop(columns=["Item_Identifier", "Outlet_Identifier"])

# Find numeric and categorical columns
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = X.select_dtypes(include=["object"]).columns

# Fill missing values
for col in numeric_cols:
    X[col] = X[col].fillna(X[col].median())

for col in categorical_cols:
    X[col] = X[col].fillna(X[col].mode()[0])

# Convert categories to numeric columns
X_encoded = pd.get_dummies(X, drop_first=True)

print("Shape after preprocessing:", X_encoded.shape)
X_encoded.head()

## 6. Train / Test Split

I split the data into training and testing parts.

The model learns from the training set and is evaluated on the test set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

## 7. Evaluation Function

I use three common regression metrics:

- **MAE**: average absolute error.
- **RMSE**: larger errors get punished more.
- **R²**: explains how much variance the model captures.

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    print(model_name)
    print("MAE:", round(mae, 2))
    print("RMSE:", round(rmse, 2))
    print("R²:", round(r2, 4))

    return {
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Predictions": predictions
    }

## 8. Baseline Model: Linear Regression

I start with Linear Regression because it is one of the simplest regression models.

In [ ]:
linear_model = LinearRegression()
linear_result = evaluate_model(
    linear_model,
    X_train,
    X_test,
    y_train,
    y_test,
    "Linear Regression"
)

## 9. Visual Check: Actual vs Predicted

A perfect model would have points close to a diagonal line.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, linear_result["Predictions"], alpha=0.4)
plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title("Linear Regression: Actual vs Predicted")
plt.show()

## 10. Cross-Validation

Cross-validation gives a more stable idea of model performance because the model is tested several times on different splits.

In [ ]:
cv_scores = cross_val_score(
    LinearRegression(),
    X_encoded,
    y,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

print("RMSE scores:", np.round(-cv_scores, 2))
print("Average RMSE:", round((-cv_scores).mean(), 2))

## 11. Second Model: Decision Tree Regressor

A Decision Tree can capture non-linear relationships, but it can also overfit if it becomes too deep.

In [ ]:
tree_model = DecisionTreeRegressor(max_depth=5, random_state=RANDOM_STATE)
tree_result = evaluate_model(
    tree_model,
    X_train,
    X_test,
    y_train,
    y_test,
    "Decision Tree (max_depth=5)"
)

## 12. Simple Hyperparameter Tuning

Here I test a few values for `max_depth` and choose the one with the lowest RMSE.

In [ ]:
depths = [3, 5, 7, 10]
tuning_results = []

for depth in depths:
    model = DecisionTreeRegressor(max_depth=depth, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    tuning_results.append({"max_depth": depth, "RMSE": rmse})

tuning_df = pd.DataFrame(tuning_results)
print(tuning_df)

best_depth = tuning_df.sort_values("RMSE").iloc[0]["max_depth"]
print("Best max_depth:", int(best_depth))

In [ ]:
best_tree_model = DecisionTreeRegressor(max_depth=int(best_depth), random_state=RANDOM_STATE)
best_tree_result = evaluate_model(
    best_tree_model,
    X_train,
    X_test,
    y_train,
    y_test,
    f"Decision Tree Tuned (max_depth={int(best_depth)})"
)

## 13. Regularization: Ridge Regression

Ridge Regression is similar to Linear Regression, but it uses regularization.

Regularization helps reduce overfitting by making the model less sensitive to individual features.

In [ ]:
ridge_model = Ridge(alpha=10)
ridge_result = evaluate_model(
    ridge_model,
    X_train,
    X_test,
    y_train,
    y_test,
    "Ridge Regression"
)

## 14. Compare the Models

I compare the models by MAE, RMSE, and R².

For MAE and RMSE, lower is better. For R², higher is better.

In [ ]:
results = pd.DataFrame([
    {k: v for k, v in linear_result.items() if k != "Predictions"},
    {k: v for k, v in tree_result.items() if k != "Predictions"},
    {k: v for k, v in best_tree_result.items() if k != "Predictions"},
    {k: v for k, v in ridge_result.items() if k != "Predictions"},
])

results = results.sort_values("RMSE")
results

## 15. Conclusion

In this exercise I tested several regression models for Big Mart sales prediction.

My main observations:

- Linear Regression is a good simple baseline.
- Decision Tree can capture more complex patterns, but it needs tuning.
- Ridge Regression adds regularization and can be useful when there are many encoded features.
- RMSE is useful here because large sales prediction errors are important.

The best model in this notebook is the one with the lowest RMSE in the comparison table.